In [2]:
# Instalación de las tres librerías necesarias
!pip install requests scapy transformers torch --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 49.9 MB/s eta 0:00:00


In [ ]:
import requests

# IP simulada sospechosa (inventada para la práctica)
ip_sospechosa = "185.220.101.45"

# Consultamos la API pública de ip-api.com (gratuita, sin token)
url = f"http://ip-api.com/json/{ip_sospechosa}"
respuesta = requests.get(url)
datos = respuesta.json()

# Mostramos la información obtenida
print("=== ANÁLISIS DE IP ===")
print(f"IP analizada : {datos.get('query')}")
print(f"País         : {datos.get('country')}")
print(f"Ciudad       : {datos.get('city')}")
print(f"Proveedor    : {datos.get('isp')}")
print(f"Estado       : {datos.get('status')}")

=== ANÁLISIS DE IP ===
IP analizada : 185.220.101.45
País         : Germany
Ciudad       : Brandenburg an der Havel
Proveedor    : Stiftung Erneuerbare Freiheit
Estado       : success


In [3]:
from scapy.all import Ether, IP, TCP, wrpcap, rdpcap
from collections import Counter

# ── Paso 1: Creamos paquetes de red simulados ──────────────────────
paquetes_simulados = [
    Ether() / IP(src="192.168.1.10", dst="185.220.101.45") / TCP(dport=80),
    Ether() / IP(src="192.168.1.10", dst="185.220.101.45") / TCP(dport=443),
    Ether() / IP(src="10.0.0.5",     dst="8.8.8.8")         / TCP(dport=53),
    Ether() / IP(src="10.0.0.5",     dst="8.8.8.8")         / TCP(dport=53),
    Ether() / IP(src="172.16.0.3",   dst="185.220.101.45") / TCP(dport=22),
    Ether() / IP(src="172.16.0.3",   dst="185.220.101.45") / TCP(dport=22),
    Ether() / IP(src="172.16.0.3",   dst="185.220.101.45") / TCP(dport=22),
    Ether() / IP(src="192.168.1.99", dst="1.1.1.1")        / TCP(dport=80),
]

# ── Paso 2: Guardamos como archivo .pcap ───────────────────────────
wrpcap("muestra.pcap", paquetes_simulados)
print("Archivo muestra.pcap creado correctamente ✓")

# ── Paso 3: Lo leemos y analizamos ────────────────────────────────
paquetes = rdpcap("muestra.pcap")
print(f"Total de paquetes capturados: {len(paquetes)}")

# Extraemos IPs de origen
ips_origen = [pkt[IP].src for pkt in paquetes if pkt.haslayer(IP)]
conteo = Counter(ips_origen).most_common(5)

print("\n=== TOP IPs DE ORIGEN ===")
for ip, cantidad in conteo:
    print(f"  {ip:<20} → {cantidad} paquetes")

# Detectamos puertos destino usados (posibles indicadores de amenaza)
puertos = [pkt[TCP].dport for pkt in paquetes if pkt.haslayer(TCP)]
print("\n=== PUERTOS DESTINO DETECTADOS ===")
for puerto, cant in Counter(puertos).most_common():
    print(f"  Puerto {puerto:<6} → {cant} paquetes")

Archivo muestra.pcap creado correctamente ✓
Total de paquetes capturados: 8

=== TOP IPs DE ORIGEN ===
  172.16.0.3           → 3 paquetes
  192.168.1.10         → 2 paquetes
  10.0.0.5             → 2 paquetes
  192.168.1.99         → 1 paquetes

=== PUERTOS DESTINO DETECTADOS ===
  Puerto 22     → 3 paquetes
  Puerto 80     → 2 paquetes
  Puerto 53     → 2 paquetes
  Puerto 443    → 1 paquetes


In [ ]:
from transformers import pipeline

# Cargamos modelo preentrenado de clasificación de sentimientos
# (usado aquí para detectar tono negativo/sospechoso en textos)
clasificador = pipeline("text-classification",
               model="distilbert-base-uncased-finetuned-sst-2-english")

# Asuntos de correo a analizar (simulados)
correos = [
    "Your account has been compromised. Click here immediately.",
    "Meeting scheduled for Thursday at 3pm. Please confirm.",
    "URGENT: Verify your password now or lose access forever!",
    "Quarterly report attached for your review.",
]

print("=== CLASIFICACIÓN DE CORREOS ===")
for texto in correos:
    resultado = clasificador(texto)[0]
    etiqueta = resultado['label']
    score = resultado['score']
    alerta = "⚠️ SOSPECHOSO" if etiqueta == "NEGATIVE" else "✅ NORMAL"
    print(f"\n{alerta} ({score:.0%})")
    print(f"  → {texto}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

=== CLASIFICACIÓN DE CORREOS ===

⚠️ SOSPECHOSO (100%)
  → Your account has been compromised. Click here immediately.

✅ NORMAL (99%)
  → Meeting scheduled for Thursday at 3pm. Please confirm.

⚠️ SOSPECHOSO (99%)
  → URGENT: Verify your password now or lose access forever!

⚠️ SOSPECHOSO (54%)
  → Quarterly report attached for your review.
